## **Setup**


In [1]:
!git clone --branch experiment/mixer-single-skip \
    https://github.com/dohyunii/VisionLab.git /content/VisionLab

%cd /content/VisionLab
%pip install -r requirements.txt

Cloning into '/content/VisionLab'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 114 (delta 54), reused 70 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 63.47 KiB | 12.69 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/VisionLab


In [2]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
GPU: True


In [3]:
from google.colab import drive
drive.mount("/content/drive")

MessageError: Error: credential propagation was unsuccessful

## **Experiment settings**

Defaults are shown in parentheses. Only `--arch` is required.

**Model and data**
- `--dataset`: `cifar10` or `cifar100` (**cifar10**)
- `--arch`: **required**
  - ResNet: `resnet18`, `resnet50`
  - PreActResNet: `preactresnet18`, `preactresnet50`
  - DenseNet: `densenet121`
  - FractalNet: `fractalnet20`, `fractalnet40`
  - VisionTransformer: `vitB4`, `vitB16`, `vitL32`
  - MLPMixer: `mlpmixerS4`, `mlpmixerB16`, `mlpmixerL32`
- `--method`: `supervised` (**supervised**)

**Classification head**
- `--head`: `linear` or `mlp` (**linear**)
- `--head_hidden_dim`: MLP hidden width (**backbone feature dimension**)
- `--head_activation`: `tanh`, `gelu`, or `relu` (**relu**)

**Training**
- `--batch_size`: images per batch (**128**)
- `--epochs`: training epochs (**100**)
- `--optimizer`: `sgd` or `adamw` (**sgd**)
- `--lr`: initial learning rate (**0.1**)
- `--momentum`: SGD momentum (**0.9**)
- `--weight_decay`: weight decay for the selected optimizer (**0.01**)

**Optimizer details**
- **SGD:** uses `--lr`, `--momentum`, and `--weight_decay`.
- **AdamW:** adaptive learning rates with decoupled weight decay; uses `--lr`
  and `--weight_decay`. Betas are `(0.9, 0.999)` and epsilon is `1e-8`.

**Execution**
- `--num_workers`: data-loading workers; `0` uses the main process (**2**)
- `--seed`: random seed (**42**)

**Storage**
- `--data_dir`: dataset directory (**<save_dir>/data** when omitted)
- `--save_dir`: experiment output directory (**./runs**)

This notebook explicitly separates persistent Drive storage:
- Dataset: `--data_dir /content/drive/MyDrive/VisionLab-data`
- Results: `--save_dir /content/drive/MyDrive/VisionLab-runs`

**Learning-rate schedule**

The learning rate is multiplied by `0.1` at the end of epochs
`int(epochs * 0.5)` and `int(epochs * 0.75)`.
For 100 epochs, these are epochs 50 and 75.

**Minimal command**

`python main.py --arch fractalnet20`


## **Run an experiment**


In [ ]:
!python -u compare_mixers.py \
    --data_dir /content/drive/MyDrive/VisionLab-data \
    --save_dir /content/drive/MyDrive/VisionLab-runs/mixer-comparison \
    --epochs 100 --seeds 42 43 44

## **Display the latest saved plot**


In [ ]:
from pathlib import Path
from IPython.display import Image, display

output_dir = Path("/content/drive/MyDrive/VisionLab-runs")
plots = list(output_dir.glob("*/training_curves.png"))

if not plots:
    print("No training plots found. Check the output path and Drive mount.")
else:
    latest_plot = max(plots, key=lambda path: path.stat().st_mtime)
    print("Experiment:", latest_plot.parent.name)
    display(Image(filename=str(latest_plot)))